# Нагрузочное тестирование сервиса

Вся логика — в модуле `load_test.py` (запускается и как скрипт из CI:
`python load_test.py`, код возврата — статус SLO). Здесь — тот же прогон
в интерактивном виде. Сервис должен быть поднят (`uvicorn app1:app --port 8079`
или docker compose). Параметры — переменными окружения: `LOAD_TEST_URL`,
`LOAD_TEST_WORKERS`, `LOAD_TEST_MAX_TIME` (секунды), `LOAD_TEST_SAMPLES`,
`LOAD_TEST_P95_SLO_MS`, `LOAD_TEST_MIN_SUCCESS_RATE`, `LOAD_TEST_DATA_PATH`,
`LOAD_TEST_FIRST_ROWS`, `LOAD_TEST_REPORT_DIR`. Отчёты пишутся в
`artifacts/load_test_report.html` / `artifacts/load_test_report.png`;
при нарушении SLO ячейка падает с `ValueError`.

In [1]:
from IPython.display import HTML, display
import pandas as pd

import load_test

label_map = load_test.load_label_classes()
profiles = load_test.load_profiles()
print(f'URL: {load_test.URL}, workers: {load_test.WORKERS}, '
      f'max_time: {load_test.MAX_TIME}с, профилей: {len(profiles)}')
print(f'SLO: p95 < {load_test.P95_SLO_MS} мс, '
      f'success_rate >= {load_test.MIN_SUCCESS_RATE}%')

URL: http://localhost:8079/predict, workers: 50, max_time: 50.0с, профилей: 100
SLO: p95 < 5000.0 мс, success_rate >= 99.0%


In [2]:
# Прогон. Для повторного/долгого прогона: LOAD_TEST_MAX_TIME=505,
# LOAD_TEST_WORKERS=50 (параметры задаются через env).
print('Starting load test...')
results = load_test.run_load_test(profiles, load_test.URL,
                                max_time=load_test.MAX_TIME,
                                workers=load_test.WORKERS,
                                label_map=label_map)
print('Test completed, generating report...')
report = load_test.generate_report(results)

display(HTML(pd.DataFrame([report]).to_html()))
display(HTML('<h2>First 5 responses:</h2>'))
display(results.head().to_html())

Starting load test...
  отправлено 50, готово 1
  отправлено 100, готово 51
  отправлено 150, готово 101
  отправлено 200, готово 151
  отправлено 250, готово 201
  отправлено 300, готово 251
  отправлено 350, готово 301
  отправлено 400, готово 351
  отправлено 450, готово 401
  отправлено 500, готово 451
  отправлено 550, готово 501
  отправлено 600, готово 551
  отправлено 650, готово 601
  отправлено 700, готово 651
  отправлено 750, готово 701
  отправлено 800, готово 751
  отправлено 850, готово 801
  отправлено 900, готово 851
  отправлено 950, готово 901
  отправлено 1000, готово 951
  отправлено 1050, готово 1001
  отправлено 1100, готово 1051
  отправлено 1150, готово 1101
  отправлено 1200, готово 1151
  отправлено 1250, готово 1201
  отправлено 1300, готово 1251
  отправлено 1350, готово 1301
  отправлено 1400, готово 1351
Test completed, generating report...
Отчёты сохранены: artifacts\load_test_report.html, artifacts\load_test_report.png


,total_requests,success_rate,avg_latency,p95_latency,p99_latency,rps,error_distribution
0,1408,99.005682,1822.454449,2483.313203,2671.183009,27.556851,{'HTTP 422': 14}


'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th></th>\n      <th>status</th>\n      <th>latency</th>\n      <th>success</th>\n      <th>error</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <th>0</th>\n      <td>422</td>\n      <td>178.670645</td>\n      <td>False</td>\n      <td>HTTP 422</td>\n    </tr>\n    <tr>\n      <th>1</th>\n      <td>200</td>\n      <td>1837.419033</td>\n      <td>True</td>\n      <td>None</td>\n    </tr>\n    <tr>\n      <th>2</th>\n      <td>200</td>\n      <td>1837.759256</td>\n      <td>True</td>\n      <td>None</td>\n    </tr>\n    <tr>\n      <th>3</th>\n      <td>200</td>\n      <td>1833.377600</td>\n      <td>True</td>\n      <td>None</td>\n    </tr>\n    <tr>\n      <th>4</th>\n      <td>200</td>\n      <td>1841.377020</td>\n      <td>True</td>\n      <td>None</td>\n    </tr>\n  </tbody>\n</table>'